# Módulo 2 · 1/3 — Trabajar con Archivos en Python

En esta clase vamos a cubrir:

1. Abrir, leer, escribir y añadir a archivos de texto
2. El módulo `os`: navegar, crear, renombrar y eliminar
3. El módulo `shutil`: copiar, mover y borrar carpetas/archivos
4. El módulo `glob`: buscar archivos por patrones
5. Trabajar con archivos CSV
6. Manejo de errores, permisos y codificación
7. Un quiz interactivo para repasar

> Los datos de ejemplo están en la carpeta `datos/` junto a este notebook.


In [ ]:
import os

In [ ]:
dir(os)

In [ ]:
dir(os.path)

---
## 1. Abrir y leer archivos de texto

La función `open()` abre un archivo y recibe dos argumentos: el **nombre** y el **modo**.

| Modo | Significado |
|------|-------------|
| `"r"` | Solo lectura (por defecto). Falla si el archivo no existe. |
| `"w"` | Escritura. Crea el archivo si no existe y **borra** su contenido si ya existe. |
| `"a"` | Añadir (append): agrega al final sin borrar. Crea si no existe. |
| `"x"` | Creación exclusiva: crea, pero falla si el archivo ya existe. |

El método `read()` lee todo el contenido. Para archivos grandes conviene leer **línea a línea**.


In [ ]:
from pathlib import Path

ruta = Path("C:/Users/Usuario")
ruta

In [ ]:
10 / 3

In [ ]:
escritorio = ruta / "Desktop" / "Mi Documento"

escritorio

In [ ]:
from pathlib import Path

ruta = Path("datos/archivos/texto/notas_01.txt")

# Lectura completa con with: se cierra el archivo automáticamente

with open(ruta, encoding="utf-8", mode="r") as f:
    contenido = f.read()
    
print(contenido)


In [ ]:
with open(ruta, encoding="utf-8", mode="r") as f:
    for linea in f:
        print(linea.strip())

In [ ]:
# Lectura línea por línea (eficiente en memoria para archivos grandes)
with open(ruta, encoding="utf-8") as f:
    lineas = [linea.rstrip() for linea in f]
print("Líneas:", lineas)

In [ ]:
with open(ruta, encoding="utf-8", mode="r") as f:
    contenido = f.readlines()

contenido

---
## 2. Escribir y añadir

- Con `"w"` **sobrescribes** todo el archivo.
- Con `"a"` **añades** al final sin borrar nada.
- Usamos `\n` para separar líneas.


In [ ]:
from pathlib import Path

ruta = Path("datos/archivos/texto/salida_2.txt")

try:
    with open(ruta, "x", encoding="utf-8") as f:
        f.write("Primera línea 1 \n")
        f.write("Segunda línea\n")
except FileExistsError:
    print(f"El archivo {ruta} ya existe. No se puede crear.")


In [ ]:
with open(ruta, "a", encoding="utf-8") as f:
    f.write("Línea añadida al final\n")

print( open( ruta, encoding="utf-8").read())

In [ ]:
help(print)

In [ ]:
arc = open(ruta, "a", encoding="utf-8")

print("Línea añadida al final con print", file=arc)

print("Línea añadida al final con print", file=arc)

arc.close()

print(arc.closed)

print(open(ruta, "r", encoding="utf-8").read())


---
## 3. El módulo `os`

`os` es la línea directa con el sistema operativo para trabajar sobre archivos y directorios.

- `os.getcwd()` → directorio de trabajo actual.
- `os.listdir(ruta)` → lista los archivos/carpetas de un directorio.
- `os.mkdir()` / `os.rmdir()` → crear / eliminar carpetas (vacías).
- `os.rename(origen, destino)` → renombrar o mover.
- `os.remove()` → eliminar un archivo.
- `os.stat()` → información detallada (tamaño, fecha, permisos).
- `os.path.join()`, `os.path.basename()`, `os.path.exists()` → manipular rutas.

> En Python moderno se prefiere `pathlib.Path`, que es más orientado a objetos y portable.


In [ ]:
import os
from datetime import datetime

print("Directorio actual:", os.getcwd())
print()
print("Contenido de datos/archivos/texto:")
for nombre in os.listdir("datos/archivos/texto"):
    print("  -", nombre)
print()

# Información de un archivo con os.stat
st = os.stat("datos/archivos/texto/notas_01.txt")
print("Tamaño (bytes):", st)
print("Tamaño (bytes):", st.st_size)
print("Última modificación:", datetime.fromtimestamp(st.st_mtime))

---
## 4. El módulo `shutil`

`shutil` ofrece atajos de alto nivel para las tareas más comunes:

- `shutil.copy(origen, destino)` → copia un archivo.
- `shutil.copytree(origen, destino)` → copia una carpeta y todo su contenido (recursivo).
- `shutil.move(origen, destino)` → mueve un archivo o carpeta.
- `shutil.rmtree(ruta)` → borra un directorio **y todo** su contenido (¡con precaución!).

Con `os.access(ruta, os.W_OK)` podemos comprobar permisos antes de operar.


In [ ]:
import shutil
from pathlib import Path

# Creamos una carpeta de destino temporal
destino = Path("datos/archivos/copia_temporal")
destino.mkdir(exist_ok=True)


In [ ]:

# Copiar un archivo de ejemplo
origen = "datos/archivos/texto/notas_01.txt"
copia = destino / "notas_copia.txt"
shutil.copy(origen, copia)
print("Copiado a:", copia, "| existe:", copia.exists())


In [ ]:

# Moverlo a una nueva ubicación
movido = Path("datos/archivos/fotos/notas_movida.txt")
shutil.move(str(copia), str(movido))
print("Movido a:", movido, "| la copia original ya no existe:", not copia.exists())


In [ ]:

# Limpiar
movido.unlink()
destino.rmdir()
print("Limpieza OK")

---
## 5. El módulo `glob`

`glob.glob(patron)` encuentra archivos que coinciden con un **patrón comodín**:

- `*.txt` → todos los `.txt` del directorio actual.
- `**/*.pdf` con `recursive=True` → todos los `.pdf` en subdirectorios.
- `*.[txt,pdf]` → archivos `.txt` **o** `.pdf`.
- `data/202*/sales_*.csv` → patrones combinados.

Ideal para **procesamiento por lotes** y para localizar archivos con características concretas.


In [ ]:
import glob

dir(glob)

In [ ]:
import glob

print("PDFs (directorio + subdirectorios):")
for p in glob.glob("datos/archivos/**/*.pdf", recursive=True):
    print("  ", p)
print()
print("JPEGs de la carpeta fotos:")
for p in glob.glob("datos/archivos/fotos/*.jpg"):
    print("  ", p)

---
## 6. Archivos CSV

CSV (valores separados por comas) guarda datos tabulares: cada fila es un registro y cada columna un campo.

El módulo `csv` permite leer y escribir estos archivos de forma sencilla:

- `csv.reader(f)` → itera fila por fila como listas.
- `csv.DictReader(f)` → lee cada fila como diccionario usando la cabecera.
- `csv.writer(f)` → escribe filas.

> Para análisis de datos tabulares, `pandas` (ver siguiente notebook) es mucho más potente.


In [ ]:
import csv

ruta = "datos/archivos/clientes_norte.csv"

print("Con reader (listas):")
with open(ruta, encoding="utf-8") as f:

    for fila in csv.reader(f):
        print("  ", fila)

print()
print("Con DictReader (diccionarios):")
with open(ruta, encoding="utf-8") as f:
    for fila in csv.DictReader(f):
        print("  ", fila["cliente"], "->", fila["email"], f'${fila["monto"]}')

---
## 7. Manejo de errores, permisos y codificación

Al trabajar con archivos pueden ocurrir problemas: el archivo no existe, no tenemos permisos o la codificación es distinta.

- **`try-except`** permite capturar y manejar errores con elegancia en lugar de que el programa se rompa.
- `PermissionError` indica falta de permisos de lectura/escritura.
- `FileNotFoundError` indica que el archivo no existe.
- La **codificación** (habitualmente UTF-8) debe indicarse al abrir cuando sea diferente.


In [ ]:
from pathlib import Path

ruta = Path("datos/archivos/inexistente.txt")

try:
    with open(ruta, encoding="utf-8") as f:
        print(f.read())
except FileNotFoundError as e:
    print("No existe:", ruta.name)
    print(e)
except PermissionError:
    print("Sin permisos para leer:", ruta.name)

---
# Quiz interactivo: Archivos

Respondé cada pregunta y pulsá **Ver respuesta** para comprobar. Hay preguntas de una sola opción y de múltiple respuesta (seleccionar todas las correctas).


In [67]:
# --- Configuración del quiz ---
# 1) Localizar la raíz del proyecto (la que contiene la carpeta 'quiz') y añadirla al sys.path
import sys
from pathlib import Path

RAIZ = Path.cwd()
while RAIZ.parent != RAIZ and not (RAIZ / "quiz").exists():
    RAIZ = RAIZ.parent
if str(RAIZ) not in sys.path:
    sys.path.append(str(RAIZ))

# 2) Importar funciones del módulo reutilizable
from quiz.quiz import aplicar_estilos, crear_quiz, crear_quiz_multiple

# 3) Aplicar el CSS (legible en VS Code y JupyterLab)
aplicar_estilos()

# 4) Cargar respuestas y explicaciones desde el JSON (misma carpeta que el notebook)
import json
with open(Path.cwd() / "Modulo_2_Archivos_preguntas.json", encoding="utf-8") as fh:
    respuestas = json.load(fh)

print("Quiz listo:", ", ".join(respuestas.keys()))

Quiz listo: q1, q2, q3, q4, q5, q6, q7, q8, q9, q10


### Pregunta 1

Para registrar visitas en un archivo `website_log.txt` *añadiendo* cada nueva entrada **sin sobrescribir** las anteriores, qué modo de apertura usarías?

In [66]:
q1_data = respuestas["q1"]
crear_quiz(q1_data["opciones"], q1_data["correcta"], q1_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) "r" (lectura)', 'A'), ('B) "w" (escritura)', 'B'), ('C…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 2

Querés **mover** un grupo de archivos a otra carpeta. ¿Qué módulo y función es el más adecuado?

In [68]:
q2_data = respuestas["q2"]
crear_quiz(q2_data["opciones"], q2_data["correcta"], q2_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) os.rename()', 'A'), ('B) shutil.move()', 'B'), ('C) gl…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 3

Qué devuelve la expresión `glob.glob("datos/archivos/**/*.pdf", recursive=True)`?

In [69]:
q3_data = respuestas["q3"]
crear_quiz(q3_data["opciones"], q3_data["correcta"], q3_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) Busca archivos .txt en el directorio actual', 'A'), ('…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 4

Necesitás localizar todos los archivos `.pdf` dispersos en subcarpetas y **moverlos** a una carpeta central. ¿Qué módulos te ayudan? (Seleccioná todas las correctas)

In [70]:
q4_data = respuestas["q4"]
crear_quiz_multiple(q4_data["opciones"], q4_data["correctas"], q4_data["explicacion"])

Checkbox(value=False, description='A) glob para localizar los archivos .pdf en subdirectorios', indent=False, …

Checkbox(value=False, description='B) shutil para mover los archivos a un nuevo directorio', indent=False, lay…

Checkbox(value=False, description='C) math para calcular tamaños', indent=False, layout=Layout(width='100%'))

Checkbox(value=False, description='D) os.path para manipular las rutas de los archivos', indent=False, layout=…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 5

¿Cuál es la principal ventaja de la sentencia `with open(...)`?

In [71]:
q5_data = respuestas["q5"]
crear_quiz(q5_data["opciones"], q5_data["correcta"], q5_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) Abre el archivo en modo escritura por defecto', 'A'), …

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 6

¿Qué modo de apertura **crea** un archivo nuevo pero **falla** si el archivo ya existe?

In [72]:
q6_data = respuestas["q6"]
crear_quiz(q6_data["opciones"], q6_data["correcta"], q6_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) "r"', 'A'), ('B) "w"', 'B'), ('C) "a"', 'C'), ('D) "x"…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 7

¿Qué función del módulo `os` devuelve el directorio de trabajo actual?

In [73]:
q7_data = respuestas["q7"]
crear_quiz(q7_data["opciones"], q7_data["correcta"], q7_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) os.getcwd()', 'A'), ('B) os.listdir()', 'B'), ('C) os.…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 8

¿Qué patrón `glob` encuentra todos los archivos `.txt` del **directorio actual**?

In [74]:
q8_data = respuestas["q8"]
crear_quiz(q8_data["opciones"], q8_data["correcta"], q8_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(("A) glob.glob('*.txt')", 'A'), ("B) glob.glob('**/*.txt', …

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 9

Un script lanza `PermissionError` al leer un CSV. ¿Cuál es la causa más probable?

In [75]:
q9_data = respuestas["q9"]
crear_quiz(q9_data["opciones"], q9_data["correcta"], q9_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) El archivo contiene datos con formato inválido', 'A'),…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 10

¿Cuál de las siguientes afirmaciones sobre `os.path` es correcta?

In [76]:
q10_data = respuestas["q10"]
crear_quiz(q10_data["opciones"], q10_data["correcta"], q10_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) os.path.join() une partes de una ruta de forma portabl…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

## ¡Fin del quiz!

Revisá las **explicaciones** de cada pregunta si fallaste alguna. Reiniciá y volvé a ejecutar la celda **setup** y las preguntas si querés reintentar.